In [2]:
import logging
import os
import time
import h5py
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import optax
from scipy.sparse.linalg import eigsh
import pickle
import numpy as np
from NES_VMC_V1 import (
    SingleStateAnsatz,
    NESTotalAnsatz,
    NESTotalAnsatz_stable,
    create_single_machine_gauge_fixed,
    Ham_Psi_scaled,
    flatten_batched_pytree,
    NESFermionHopRule,
    ravel_pytree,
)
from NES_VMC_tool import create_gauge_reset_total_machines,NES_loss_energy_stable_gauge,\
    nes_vmc_gradient_stable_gauge,make_grad_fn_gauge,make_qgt_fn_gauge,make_gauge_fn
from H2_631G import SINGLE_SIZE, ha, hi_ext, ext_edges, K, Hatree_Fock,hi,E_fcis
import logging

/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: uv is a replacement for pip which helps you follow good software practices.

H2 分子基本信息
HF energy = -0.99749729 Ha
Total electrons = (1, 1)
Total basis functions = 4
H₂ FCI 基准能量
E0 = -1.05434745 Ha  |  激发能: 0.0000 eV
E1 = -0.95790573 Ha  |  激发能: 2.6243 eV
E2 = -0.66895227 Ha  |  激发能: 10.4871 eV
E3 = -0.55140192 Ha  |  激发能: 13.6859 eV

HF reference state: [0 0 0 1 0 0 0 1]
single_edges: [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3), (4, 5), (4, 6), (4, 7), (5, 6), (5, 7), (6, 7)]


In [3]:
K = 4
total_ansatz = NESTotalAnsatz(
    n_spin_orbitals=SINGLE_SIZE,
    n_states=K,
    hidden_dim=SINGLE_SIZE + K,
    rngs=nnx.Rngs(11),
) # 总波函数拟设
single_ansatz = SingleStateAnsatz(
    n_spin_orbitals=SINGLE_SIZE,
    hidden_dim=SINGLE_SIZE + K,
    rngs=nnx.Rngs(11),
) # 单态波函数拟设

ha.to_dense().shape #shape = (4,4) 

(16, 16)

In [7]:

HISTORY_FILE = r'./data/26-09-08-18-09_history_natural_gradient_H2_molecule_K4.pkl'
with open(HISTORY_FILE, "rb") as f:
    history = pickle.load(f)

history['Energy_levels'][200]
# 检查参数是否正确


Array([-1.0463837 +0.00378942j, -0.9575794 +0.00010092j,
       -0.66382189+0.00164811j, -0.54445152+0.0001313j ],      dtype=complex128)

In [8]:
E_fcis

array([-1.05434745, -0.95790573, -0.66895227, -0.55140192])

In [ ]:
finnal_params = history['params'][-1]
single_graphdef,single_params = nnx.split(total_ansatz.single_ansatz_list[0])
fine_single_ansatz = nnx.merge(single_graphdef,finnal_params['single_ansatz_list'][0])

In [ ]:
fine_single_ansatz(hi.all_states()) #psi(x) 

In [ ]:
fine_single_ansatz(hi.all_states()) #psi(x)  shape=(16,)
ha.to_dense().shape #shape = (4,4)

In [ ]:
wavefunction = np.exp(fine_single_ansatz(hi.all_states()))
wavefunction

In [ ]:
normalization = wavefunction.conj().T@wavefunction
normalization

In [ ]:
wavefunction.conj().T@ha.to_dense()@wavefunction/normalization

In [ ]:
history['Energy_levels'][-1]